# Proyecto de Base de Datos y Dashboard

Este notebook presenta la primera fase del proyecto, enfocada en la **carga, comprensión, limpieza y transformación** del dataset de **Mesas de Ayuda**.

La fuente del dataset proviene de un archivo CSV consultado desde una fuente de datos abierta, y contiene información agregada sobre solicitudes atendidas en una mesa de ayuda. Su utilidad dentro del proyecto es permitir el análisis de categorías de soporte, volumen de casos, tiempos promedio de solución y nivel de cierre o resolución de solicitudes.

A partir de esta fuente se desarrolla un flujo de trabajo que incluye:

- carga del dataset,
- inspección inicial de la información,
- limpieza y estandarización de columnas,
- separación de categorías de solicitud,
- transformación del tiempo promedio a formato numérico,
- construcción de una tabla dimensión y una tabla de hechos,
- exportación de archivos listos para la siguiente fase.

Esta primera entrega deja organizada la base del proyecto para continuar con el **MER**, el **modelo relacional**, las **consultas SQL** y el **dashboard en Streamlit**.

In [ ]:
# Importar librerías

import pandas as pd
import numpy as np

url = "https://www.datos.gov.co/api/views/ja3t-iydp/rows.csv"

df = pd.read_csv(url) # leer datos desde internet
df.to_csv("Mesas_de_ayuda.csv", index=False, encoding="utf-8") # guardar esos datos como archivo dentro de Colab

print("Dataset descargado correctamente")
print("Archivo guardado como: Mesas_de_ayuda.csv")
print("Dimensiones:", df.shape)

df.head()

Dataset descargado correctamente
Archivo guardado como: Mesas_de_ayuda.csv
Dimensiones: (63, 7)


,Año,TIPO de Solictud,Número de casos abiertos,Número de casos resueltos,Número de casos atrasados,Número de casos cerrados,Tiempo promedio de solución
0,2025,Acceso,1,1,0,1,00-03-23-53
1,2025,Acceso > Acceso remoto por VPN,2,2,0,2,06-06-45-46
2,2025,Acceso > CARPETA COMPARTIDA,35,35,0,35,01-21-26-04
3,2025,Acceso > CORREO ELECTRÓNICO,127,127,0,127,00-06-52-36
4,2025,Acceso > KACTUS,76,76,0,76,00-05-36-56


In [ ]:
# Inspección inicial del DataFrame

print("Columnas:")
print(df.columns.tolist())

print("\nTipos de datos:")
print(df.dtypes)

print("\nNulos por columna:")
print(df.isnull().sum())

df.head()

Columnas:
['Año', 'TIPO de Solictud', 'Número de casos abiertos', 'Número de casos resueltos', 'Número de casos atrasados', 'Número de casos cerrados', 'Tiempo promedio de solución']

Tipos de datos:
Año                             int64
TIPO de Solictud               object
Número de casos abiertos        int64
Número de casos resueltos       int64
Número de casos atrasados       int64
Número de casos cerrados        int64
Tiempo promedio de solución    object
dtype: object

Nulos por columna:
Año                            0
TIPO de Solictud               0
Número de casos abiertos       0
Número de casos resueltos      0
Número de casos atrasados      0
Número de casos cerrados       0
Tiempo promedio de solución    0
dtype: int64


,Año,TIPO de Solictud,Número de casos abiertos,Número de casos resueltos,Número de casos atrasados,Número de casos cerrados,Tiempo promedio de solución
0,2025,Acceso,1,1,0,1,00-03-23-53
1,2025,Acceso > Acceso remoto por VPN,2,2,0,2,06-06-45-46
2,2025,Acceso > CARPETA COMPARTIDA,35,35,0,35,01-21-26-04
3,2025,Acceso > CORREO ELECTRÓNICO,127,127,0,127,00-06-52-36
4,2025,Acceso > KACTUS,76,76,0,76,00-05-36-56


In [ ]:
# Limpieza: Renombrado de columnas

df = df.rename(columns={
    "Año": "anio",
    "TIPO de Solictud": "tipo_solicitud",
    "Número de casos abiertos": "casos_abiertos",
    "Número de casos resueltos": "casos_resueltos",
    "Número de casos atrasados": "casos_atrasados",
    "Número de casos cerrados": "casos_cerrados",
    "Tiempo promedio de solución": "tiempo_promedio"
})

df.head()

,anio,tipo_solicitud,casos_abiertos,casos_resueltos,casos_atrasados,casos_cerrados,tiempo_promedio
0,2025,Acceso,1,1,0,1,00-03-23-53
1,2025,Acceso > Acceso remoto por VPN,2,2,0,2,06-06-45-46
2,2025,Acceso > CARPETA COMPARTIDA,35,35,0,35,01-21-26-04
3,2025,Acceso > CORREO ELECTRÓNICO,127,127,0,127,00-06-52-36
4,2025,Acceso > KACTUS,76,76,0,76,00-05-36-56


In [ ]:
# Transformación
# Separar tipo_principal, subtipo, tipo_completo

partes = df["tipo_solicitud"].astype(str).str.split(">", n=1, expand=True)

df["tipo_principal"] = partes[0].str.strip()
df["subtipo"] = partes[1].fillna("").str.strip()
df["tipo_completo"] = df["tipo_solicitud"].astype(str).str.strip()

df[["tipo_solicitud", "tipo_principal", "subtipo"]].head(10)

,tipo_solicitud,tipo_principal,subtipo
0,Acceso,Acceso,
1,Acceso > Acceso remoto por VPN,Acceso,Acceso remoto por VPN
2,Acceso > CARPETA COMPARTIDA,Acceso,CARPETA COMPARTIDA
3,Acceso > CORREO ELECTRÓNICO,Acceso,CORREO ELECTRÓNICO
4,Acceso > KACTUS,Acceso,KACTUS
5,Acceso > Recuperación de Contraseña,Acceso,Recuperación de Contraseña
6,Acceso > SEVEN,Acceso,SEVEN
7,Acceso > SGI,Acceso,SGI
8,Acceso > SICAU,Acceso,SICAU
9,Acceso > USUARIO DE EQUIPO,Acceso,USUARIO DE EQUIPO


In [ ]:
# Convertir tiempo a segundos

tiempo_partes = df["tiempo_promedio"].str.split("-", expand=True)

df["dias"] = tiempo_partes[0].astype(int)
df["horas"] = tiempo_partes[1].astype(int)
df["minutos"] = tiempo_partes[2].astype(int)
df["segundos"] = tiempo_partes[3].astype(int)

df["tiempo_promedio_segundos"] = (
    df["dias"] * 86400 +
    df["horas"] * 3600 +
    df["minutos"] * 60 +
    df["segundos"]
)

df[["tiempo_promedio", "tiempo_promedio_segundos"]].head()

,tiempo_promedio,tiempo_promedio_segundos
0,00-03-23-53,12233
1,06-06-45-46,542746
2,01-21-26-04,163564
3,00-06-52-36,24756
4,00-05-36-56,20216


In [ ]:
# Validación rápida

print("Total casos abiertos:", df["casos_abiertos"].sum())
print("Total casos resueltos:", df["casos_resueltos"].sum())
print("Total casos atrasados:", df["casos_atrasados"].sum())
print("Total casos cerrados:", df["casos_cerrados"].sum())

Total casos abiertos: 2109
Total casos resueltos: 2106
Total casos atrasados: 0
Total casos cerrados: 2106


In [ ]:
# Crear la dimensión "dim_tipo_solicitud"
# Esta tabla guardará las categorías únicas de solicitud

# toma solo las columnas categóricas,
# elimina duplicados,
# crea una llave artificial tipo_id,
# deja una tabla limpia para usar como dimensión

dim_tipo = (
    df[["tipo_principal", "subtipo", "tipo_completo"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

dim_tipo["tipo_id"] = range(1, len(dim_tipo) + 1)

dim_tipo = dim_tipo[["tipo_id", "tipo_principal", "subtipo", "tipo_completo"]]

print("Dimensión creada correctamente")
print("Filas, columnas:", dim_tipo.shape)
dim_tipo.head(10)

Dimensión creada correctamente
Filas, columnas: (63, 4)


,tipo_id,tipo_principal,subtipo,tipo_completo
0,1,Acceso,,Acceso
1,2,Acceso,Acceso remoto por VPN,Acceso > Acceso remoto por VPN
2,3,Acceso,CARPETA COMPARTIDA,Acceso > CARPETA COMPARTIDA
3,4,Acceso,CORREO ELECTRÓNICO,Acceso > CORREO ELECTRÓNICO
4,5,Acceso,KACTUS,Acceso > KACTUS
5,6,Acceso,Recuperación de Contraseña,Acceso > Recuperación de Contraseña
6,7,Acceso,SEVEN,Acceso > SEVEN
7,8,Acceso,SGI,Acceso > SGI
8,9,Acceso,SICAU,Acceso > SICAU
9,10,Acceso,USUARIO DE EQUIPO,Acceso > USUARIO DE EQUIPO


In [ ]:
# Verificar si la dimensión quedó bien
# Ayuda a comprobar que no haya categorías repetidas raras

print("Número de categorías únicas:", dim_tipo["tipo_id"].nunique())
print("\nTipos principales únicos:")
print(dim_tipo["tipo_principal"].value_counts())

Número de categorías únicas: 63

Tipos principales únicos:
tipo_principal
Software                 32
Acceso                   10
Dispositivos              6
Redes                     5
Acompañamiento            1
Chapas                    1
Correo electrónico        1
Concepto Técnico          1
Otros                     1
Mantenimiento             1
Pascual Bravo SAI         1
Servidor MOODLE           1
Traslados definitivos     1
Traslados Temporales      1
Name: count, dtype: int64


In [ ]:
# Crear la tabla de hechos "fact_mesas_ayuda"
# Ahora unimos df con dim_tipo para reemplazar la categoría textual por tipo_id

# busca el tipo_id correspondiente a cada fila,
# crea registro_id,
# arma la tabla final que va a MySQL.

fact = df.merge(
    dim_tipo,
    on=["tipo_principal", "subtipo", "tipo_completo"],
    how="left"
)

fact["registro_id"] = range(1, len(fact) + 1)

fact_mesas_ayuda = fact[[
    "registro_id",
    "anio",
    "tipo_id",
    "casos_abiertos",
    "casos_resueltos",
    "casos_atrasados",
    "casos_cerrados",
    "tiempo_promedio_segundos"
]].copy()

print("Tabla de hechos creada correctamente")
print("Filas, columnas:", fact_mesas_ayuda.shape)
fact_mesas_ayuda.head()

Tabla de hechos creada correctamente
Filas, columnas: (63, 8)


,registro_id,anio,tipo_id,casos_abiertos,casos_resueltos,casos_atrasados,casos_cerrados,tiempo_promedio_segundos
0,1,2025,1,1,1,0,1,12233
1,2,2025,2,2,2,0,2,542746
2,3,2025,3,35,35,0,35,163564
3,4,2025,4,127,127,0,127,24756
4,5,2025,5,76,76,0,76,20216


In [ ]:
# Validar que no se perdieron filas
# Esto es clave. Debe seguir habiendo el mismo número de filas del dataset original

print("Filas en df original:", len(df))
print("Filas en fact_mesas_ayuda:", len(fact_mesas_ayuda))

print("\nValores nulos en tipo_id:")
print(fact_mesas_ayuda["tipo_id"].isnull().sum())

Filas en df original: 63
Filas en fact_mesas_ayuda: 63

Valores nulos en tipo_id:
0


In [ ]:
# Validar totales después de la transformación
# Queremos comprobar que el modelo nuevo conserva los números del dataset

print("Total casos abiertos:", fact_mesas_ayuda["casos_abiertos"].sum())
print("Total casos resueltos:", fact_mesas_ayuda["casos_resueltos"].sum())
print("Total casos atrasados:", fact_mesas_ayuda["casos_atrasados"].sum())
print("Total casos cerrados:", fact_mesas_ayuda["casos_cerrados"].sum())

Total casos abiertos: 2109
Total casos resueltos: 2106
Total casos atrasados: 0
Total casos cerrados: 2106


In [ ]:
# Hacer una verificación analítica útil: cuáles áreas concentran más casos
# Saca una vista que luego servirá en el dashboard.

resumen_tipo_principal = (
    fact.groupby("tipo_principal", as_index=False)[
        ["casos_abiertos", "casos_resueltos", "casos_cerrados"]
    ]
    .sum()
    .sort_values("casos_abiertos", ascending=False)
)

resumen_tipo_principal

,tipo_principal,casos_abiertos,casos_resueltos,casos_cerrados
0,Acceso,1196,1196,1196
11,Software,357,354,354
5,Dispositivos,193,193,193
9,Redes,167,166,166
1,Acompañamiento,61,62,62
3,Concepto Técnico,52,52,52
6,Mantenimiento,51,51,51
4,Correo electrónico,21,21,21
7,Otros,10,10,10
2,Chapas,1,1,1


In [ ]:
# Top subtipos con más casos abiertos
# Esto luego puede convertirse en una gráfica de barras en Streamlit

top_subtipos = (
    fact.groupby("tipo_completo", as_index=False)["casos_abiertos"]
    .sum()
    .sort_values("casos_abiertos", ascending=False)
)

top_subtipos.head(10)

,tipo_completo,casos_abiertos
8,Acceso > SICAU,663
6,Acceso > SEVEN,193
29,Software,129
3,Acceso > CORREO ELECTRÓNICO,127
9,Acceso > USUARIO DE EQUIPO,93
17,Dispositivos > Impresoras,92
4,Acceso > KACTUS,76
10,Acompañamiento,61
34,Software > Instalar Programa,58
12,Concepto Técnico,52


In [ ]:
# Exportar los archivos listos para MySQL
# Ahora sí guardamos los productos útiles del notebook

df.to_csv("mesas_ayuda_limpio.csv", index=False, encoding="utf-8")
dim_tipo.to_csv("dim_tipo_solicitud.csv", index=False, encoding="utf-8")
fact_mesas_ayuda.to_csv("fact_mesas_ayuda.csv", index=False, encoding="utf-8")

print("Archivos exportados correctamente:")
print("- mesas_ayuda_limpio.csv")
print("- dim_tipo_solicitud.csv")
print("- fact_mesas_ayuda.csv")

Archivos exportados correctamente:
- mesas_ayuda_limpio.csv
- dim_tipo_solicitud.csv
- fact_mesas_ayuda.csv


In [ ]:
# Descargar los archivos desde Colab
# Si quieres bajarlos a tu computador de una vez

from google.colab import files

files.download("mesas_ayuda_limpio.csv")
files.download("dim_tipo_solicitud.csv")
files.download("fact_mesas_ayuda.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>